In [ ]:
import os
import re
import glob
import numpy as np
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Mount Google Drive

from google.colab import drive
## PLEASE UPLOAD THE SAVE DIR
drive.mount('/content/drive')
DRIVE_SAVE_DIR = "/content/drive/MyDrive/deeponet_runs"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

## 2. Data path


## PLEASE UPLOAD THE DATA PATH
## PLEASE UPLOAD THE DATA PATH
DATA_GLOB = "/content/drive/MyDrive/donnee_deep1/4_pillars/*.txt"
## PLEASE UPLOAD THE DATA PATH
## PLEASE UPLOAD THE DATA PATH

all_files = sorted(glob.glob(DATA_GLOB))
print("Number of files found:", len(all_files))
print("===================")
assert len(all_files) > 0, "No .txt files found in /content"

## 3. Fixed geometry parameters

X_ROW1 = 75e-6    # fixed x-position of row 1

### Hyperparameters

EPOCHS = 2001

DOT_DIM = 50

HIDDEN  = [100, 100, 100, 100, 100]

N_QUERY = 2048

BATCH_SIZE = 31 # simulation number

LR = 1e-3 #cos scheduler on

W_DATA = 1
W_WALL = 5

print(f"EPOCHS: {EPOCHS}")
print(f"DOT_DIM: {DOT_DIM} x2 because u and v")
print(f"HIDDEN: {HIDDEN}")
print(f"LR: {LR} + cos scheduler")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"N_QUERY: {N_QUERY}")
print(" ")
print(f"W_DATA: {W_DATA}")

print(f"W_WALL: {W_WALL}")
print("===================")
print(" ")
print("Loading files")

## 4. File parsing

def extract_params_from_filename(path):
    """
    Parses: dldtxt_r1_..._r2_..._y1_..._y2_..._y3_..._y4_..._x1_....txt
    Returns: theta (7,) = [r1, r2, y1, y2, y3, y4, x1]
    x1 = distance from 0 to row 2 (variable)
    """
    stem = os.path.splitext(os.path.basename(path))[0]


    pattern = r"{key}_(-?[0-9]+(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?)"

    def grab(key):
        m = re.search(pattern.format(key=key), stem)
        if m is None:
            raise ValueError(f"Could not parse '{key}' from: {stem}")
        return float(m.group(1))

    r1,r2 = grab("r1"), grab("r2")
    y1,y2 = grab("y1"), grab("y2")
    y3,y4 = grab("y3"), grab("y4")
    x1 = grab("x1")

    return torch.tensor([r1, r2, y1, y2, y3, y4, x1], dtype=torch.float32)


def load_comsol_txt(path):
    """
    Loads 8-column COMSOL export.
    Uses only columns 0-3: x, y, u, v.
    Columns 4-7 (ux, uy, vx, vy) are not used here
    """
    data = np.loadtxt(path, comments="%") # (M, 8)

    xy = torch.tensor(data[:, 0:2], dtype=torch.float32)
    uv = torch.tensor(data[:, 2:4], dtype=torch.float32)
    gradu = torch.tensor(data[:, 4:6], dtype=torch.float32)
    graduv = torch.tensor(data[:, 6:8], dtype=torch.float32)

    return xy, uv, gradu, graduv


## 5. Global scales

def infer_global_scales(file_list):
    max_x = max_y = max_vel = max_r = 0.0
    for path in file_list:
        xy, uv, _, _ = load_comsol_txt(path)
        max_x = max(max_x, float(xy[:, 0].max()))
        max_y = max(max_y, float(xy[:, 1].max()))
        max_vel = max(max_vel,float(torch.abs(uv).max()))
        theta= extract_params_from_filename(path)
        max_r = max(max_r, float(theta[0]), float(theta[1]))
    return max_x, max_y, max_vel, max_r

L_x, L_y, VEL_SCALE, R_SCALE = infer_global_scales(all_files)
VEL_SCALE = max(VEL_SCALE, 1e-12)
R_SCALE = max(R_SCALE,1e-12)
X1_SCALE = max(extract_params_from_filename(f)[6].item()for f in all_files)
X1_SCALE = max(X1_SCALE, 1e-12)

print(f"X1_SCALE : {X1_SCALE:.3e} m")
print(" ")
print(f"L_x: {L_x}")
print(f"L_y: {L_y}")
print(f"VEL_SCALE: {VEL_SCALE}")
print("===================")
print(" ")
print("File Loading")

## 6. Geometry: SDF for pillars

def compute_sdf_four_pillars(xy, theta):

    """
    theta = [r1, r2, y1, y2, y3, y4, x1]

      Row 1: x = X_ROW1 (fixed), pillars at y1, y2, radius r1
      Row 2: x = x1 (variable), pillars at y3, y4, radius r2
    """
    r1,r2 = theta[0], theta[1]
    y1,y2 = theta[2], theta[3]
    y3,y4 = theta[4], theta[5]
    x1 = theta[6]
    x = xy[:, 0]
    y = xy[:, 1]
    d1 = torch.sqrt((x - X_ROW1)**2 + (y - y1)**2) - r1
    d2 = torch.sqrt((x - X_ROW1)**2 + (y - y2)**2) - r1
    d3 = torch.sqrt((x - x1)**2 + (y - y3)**2) - r2
    d4 = torch.sqrt((x - x1)**2+ (y - y4)**2) - r2

    return torch.minimum(torch.minimum(d1, d2), torch.minimum(d3, d4))  # (M,)

## 7. Normalization

def normalise_theta(theta):
    """
    theta = [r1, r2, y1, y2, y3, y4, x1]
    Each component divided by its own scale
    """
    t = theta.clone()
    t[0]/= R_SCALE   # r1
    t[1]/= R_SCALE   # r2
    t[2]/= L_y       # y1
    t[3]/= L_y      # y2
    t[4]/= L_y      # y3
    t[5]/= L_y       # y4
    t[6] = (t[6] - 110e-6)/X1_SCALE
    return t # (7,)

def normalise_xy(xy):
    out = xy.clone()
    out[:, 0]/= L_x
    out[:, 1]/= L_y
    return out

def normalise_uv(uv): return uv / VEL_SCALE


## 8. Dataset

class PillarDataset(Dataset):
    def __init__(self, file_list, n_query):
        self.n_query = n_query
        self.samples = []
        print(f"Loading {len(file_list)} file(s)...")
        for path in file_list:
            theta = extract_params_from_filename(path)
            xy, uv, _, _ = load_comsol_txt(path)
            sdf = compute_sdf_four_pillars(xy, theta)
            self.samples.append({"path": path, "theta": theta,
                                  "xy": xy, "uv": uv, "sdf": sdf})
        print("Dataset ready")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
      s = self.samples[idx]
      theta, xy, uv, sdf = s["theta"], s["xy"], s["uv"], s["sdf"]
      M = xy.shape[0]
      if self.n_query is not None:
          ids = (torch.randperm(M)[:self.n_query] if M >= self.n_query
                else torch.randint(0, M, (self.n_query,)))
          xy, uv, sdf = xy[ids], uv[ids], sdf[ids]
      return {"theta": theta, "xy": xy, "uv": uv, "sdf": sdf}

def collate_fn(batch):
    return (torch.stack([b["theta"] for b in batch]),
            torch.stack([b["xy"] for b in batch]),
          torch.stack([b["uv"] for b in batch]),
           torch.stack([b["sdf"] for b in batch]),)

## 9. Train / validation / test split

files = all_files.copy()
np.random.seed(0)
np.random.shuffle(files)
n_test = math.ceil(0.26 * len(files))
n_val= math.ceil(0.22* (len(files) - n_test))

test_files = files[:n_test]
val_files = files[n_test : n_test + n_val]
train_files = files[n_test + n_val:]

print("Train files:", len(train_files))
print("Val  files :", len(val_files))
print("Test files :", len(test_files))

train_dataset = PillarDataset(train_files, N_QUERY)
val_dataset = PillarDataset(val_files, N_QUERY)
test_dataset = PillarDataset(test_files, N_QUERY)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, BATCH_SIZE,shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset,BATCH_SIZE,shuffle=False, collate_fn=collate_fn)

## 10. DeepONet architecture

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden, act=nn.Tanh):
        super().__init__()
        layers, p = [], in_dim
        for h in hidden:
            layers += [nn.Linear(p, h), act()]
            p = h
        layers += [nn.Linear(p, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class TwoPillarDeepONet(nn.Module):
    """
    B: batch_size
    M: number of points per simulation (n_query)
    Branch input: theta_norm (B, 7) = [r1, r2, y1, y2, y3, y4, x1]
    Trunk  input : trunk_input (B, M, 3) = [x, y, sdf]
    Output : uv_pred (B, M, 2)
    """

    def __init__(self, dot_dim, hidden):
        super().__init__()
        self.dot_dim = dot_dim
        self.branch = MLP(7, 2*dot_dim, hidden, nn.Tanh)
        self.trunk= MLP(3, 2*dot_dim, hidden, nn.Tanh)
        self.bias_u = nn.Parameter(torch.zeros(1))
        self.bias_v = nn.Parameter(torch.zeros(1))

    def forward(self, theta_norm, trunk_input):
        B, M, _ = trunk_input.shape
        b = self.branch(theta_norm)                          # (B, 2DOT_DIM)
        t = self.trunk(trunk_input.reshape(-1, 3))           # (B*M, 2DOT_DIM)
        t = t.reshape(B, M, 2*self.dot_dim)                  # (B, M, 2DOT_DIM)

        p = self.dot_dim
        u = (t[:, :, :p] * b[:, :p].unsqueeze(1)).sum(-1) + self.bias_u  # (B, M)
        v = (t[:, :, p:] * b[:, p:].unsqueeze(1)).sum(-1) + self.bias_v  # (B, M)
        return torch.stack([u, v], dim=-1)                   # (B, M, 2)

## 11. Losses

def loss_data(uv_pred, uv_true, sdf_norm, lambda_b=5.0, delta=0.025):

    """Weighted MSE! higher weight near pillar surfaces! """

    w = (1.0 + lambda_b * torch.exp(-torch.abs(sdf_norm) / delta)).unsqueeze(-1)
    return (w * (uv_pred - uv_true)**2).mean()


def loss_bc_walls(model, theta_raw, B, n_wall=128):

    """No-slip on top/bottom channel walls: u=v=0"""

    xs = np.linspace(0, L_x, n_wall)
    pts = np.concatenate([
        np.stack([xs, np.full_like(xs, 1e-9)],       axis=1),
        np.stack([xs, np.full_like(xs, L_y - 1e-9)], axis=1), ], axis=0)
    wall_pts = torch.tensor(pts, dtype=torch.float32, device=device)
    sdf = compute_sdf_four_pillars(wall_pts, theta_raw[0]) / L_y
    theta_n = normalise_theta(theta_raw[0]).unsqueeze(0).repeat(B, 1)
    trunk = torch.cat([normalise_xy(wall_pts), sdf.unsqueeze(-1)], dim=-1)
    trunk = trunk.unsqueeze(0).repeat(B, 1, 1)
    return (model(theta_n, trunk)**2).mean()

# Used for plot titles
H_STR    = f"{HIDDEN[0]}x{len(HIDDEN)}"
RUN_NAME = (
    f"ep{EPOCHS}"
    f"_dot{DOT_DIM}"
    f"_h{H_STR}"
    f"_lr{LR:.0e}"
    f"_bs{BATCH_SIZE}"
    f"_nq{N_QUERY}"
    f"_wD{W_DATA}_wW{W_WALL}")

## 13. Model / optimizer

model= TwoPillarDeepONet(dot_dim=DOT_DIM, hidden=HIDDEN).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
history = {"train": [], "val": [], "data": [], "wall": []}

# Live plot setup # CLAUDE CODED
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
param_str = f"dot={DOT_DIM} | h={H_STR} | lr={LR:.0e} | bs={BATCH_SIZE} | nq={N_QUERY}\n"f"wD={W_DATA}| wW={W_WALL}"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(param_str, fontsize=8)
plt.tight_layout()

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS,eta_min=1e-5)

## 14. Training loop

for epoch in range(1, EPOCHS+1):
    model.train()
    t_loss = t_wall = t_data = 0.0

    for theta_raw,xy_raw, uv_raw, sdf_raw in train_loader:
        theta_raw, xy_raw = theta_raw.to(device), xy_raw.to(device)
        uv_raw,sdf_raw  = uv_raw.to(device), sdf_raw.to(device)

        theta_norm = torch.stack([normalise_theta(t) for t in theta_raw])
        xy_norm  = torch.stack([normalise_xy(x) for x in xy_raw])
        uv_norm = normalise_uv(uv_raw)
        sdf_norm  = sdf_raw / L_y

        trunk_input = torch.cat([xy_norm, sdf_norm.unsqueeze(-1)], dim=-1)

        optimizer.zero_grad()
        uv_pred = model(theta_norm, trunk_input)

        l_data = loss_data(uv_pred, uv_norm, sdf_norm)

        l_wall = loss_bc_walls(model, theta_raw, theta_raw.shape[0])

        total = W_DATA*l_data +  W_WALL*l_wall
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        t_loss +=total.item()
        t_data += l_data.item()
        t_wall += l_wall.item()

    n = len(train_loader)
    avg_train = t_loss/n;
    avg_wall = t_wall/n; avg_data = t_data/n


    # Validation

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for theta_raw, xy_raw, uv_raw, sdf_raw in val_loader:
            theta_raw, xy_raw = theta_raw.to(device), xy_raw.to(device)
            uv_raw,  sdf_raw  = uv_raw.to(device),   sdf_raw.to(device)

            theta_norm = torch.stack([normalise_theta(t) for t in theta_raw])
            xy_norm = torch.stack([normalise_xy(x) for x in xy_raw])
            uv_norm = normalise_uv(uv_raw)
            sdf_norm  = sdf_raw / L_y

            trunk_input = torch.cat([xy_norm, sdf_norm.unsqueeze(-1)], dim=-1)
            uv_pred = model(theta_norm, trunk_input)
            v_loss += loss_data(uv_pred, uv_norm, sdf_norm).item()

    avg_val = v_loss / len(val_loader)

    history["train"].append(avg_train); history["val"].append(avg_val); history["wall"].append(avg_wall); history["data"].append(avg_data)
    scheduler.step()

    if (epoch == 1) or (epoch % 100 == 0): # CLAUDE CODED

      ax1.cla()
      ax1.semilogy(history["train"], label="train (total)")
      ax1.semilogy(history["val"],   label="test (data loss)", linestyle="--")
      ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
      ax1.set_title("Train vs. Val loss", fontsize=9)
      ax1.grid(True); ax1.legend(fontsize=8)

      ax2.cla()
      ax2.semilogy(history["data"], label="data")
      ax2.semilogy(history["wall"], label="bc walls")
      ax2.set_xlabel("epoch"); ax2.set_ylabel("loss")
      ax2.set_title("Physics losses", fontsize=9)
      ax2.grid(True); ax2.legend(fontsize=8)

      clear_output(wait=True)
      display(fig)

      print(f"Epoch {epoch:4d}/{EPOCHS} | "
            f"Train={avg_train:.3e} | Test={avg_val:.3e} | "
            f"Data={avg_data:.3e} | Wall={avg_wall:.3e}")


## 15. Learning curves # CLAUDE CODED

clear_output(wait=True)

param_str = f"dot={DOT_DIM} | h={H_STR} | lr={LR:.0e} | bs={BATCH_SIZE} | nq={N_QUERY}\n"f"wD={W_DATA}| wW={W_WALL} | "

plt.figure(figsize=(13, 4))

plt.subplot(1, 2, 1)
plt.semilogy(history["train"], label="train (data + bc)")
plt.semilogy(history["val"],   label="validation (data loss)", linestyle="--")
plt.xlabel("epoch");  plt.ylabel("loss")
plt.title(f"Train/Validation loss\n{param_str}", fontsize=9)
plt.grid(True);  plt.legend()

plt.subplot(1, 2, 2)
plt.semilogy(history["data"],  label="data")
plt.semilogy(history["wall"], label="bc walls")
plt.xlabel("epoch");  plt.ylabel("loss")
plt.title(f"Physics losses\n{param_str}", fontsize=9)
plt.grid(True);  plt.legend()

plt.tight_layout()

plt.savefig(os.path.join(DRIVE_SAVE_DIR, f"loss_{RUN_NAME}.png"), dpi=150)
plt.show()

## 16. Inference helper # CLAUDE CODED

def predict_on_file(model, path):
    """
    Predict on all mesh points of one simulation.
    Returns: xy (M,2), uv_true (M,2), uv_pred (M,2), theta (7,)
    """

    model.eval()
    theta = extract_params_from_filename(path).to(device)
    xy, uv_true, _, _ = load_comsol_txt(path)
    xy = xy.to(device);  uv_true = uv_true.to(device)

    sdf        = compute_sdf_four_pillars(xy, theta)
    theta_norm = normalise_theta(theta).unsqueeze(0)
    xy_norm    = normalise_xy(xy).unsqueeze(0)
    sdf_norm   = (sdf / L_y).unsqueeze(0).unsqueeze(-1)
    trunk      = torch.cat([xy_norm, sdf_norm], dim=-1)

    with torch.no_grad():
        uv_pred = model(theta_norm, trunk)[0] * VEL_SCALE

    return xy.cpu(), uv_true.cpu(), uv_pred.cpu(), theta.cpu()

## 17. Prediction plots # CLAUDE CODED

def plot_prediction(xy, uv_true, uv_pred, theta, title="", save_path=None):

    """
    3-panel figure: True u | Predicted u | |error u|
    """
    x_np  = 1e6*xy[:, 0].numpy()
    y_np  = 1e6*xy[:, 1].numpy()
    u_true = 1e3*uv_true[:, 0].numpy()
    u_pred = 1e3*uv_pred[:, 0].numpy()
    r1_p, r2_p = float(theta[0]), float(theta[1])
    y1_p, y2_p = float(theta[2]), float(theta[3])
    y3_p, y4_p = float(theta[4]), float(theta[5])
    x1_p       = float(theta[6])

    def add_pillars(ax):
        for (cx, cy, r) in [(X_ROW1, y1_p, r1_p), (X_ROW1, y2_p, r1_p),(x1_p,y3_p, r2_p), (x1_p,y4_p, r2_p)]:
            ax.add_patch(plt.Circle((cx*1e6, cy*1e6), r*1e6, color='white', ec='k', lw=1.0))
        ax.set_aspect('equal')

    vmin_u = min(u_true.min(), u_pred.min())
    vmax_u = max(u_true.max(), u_pred.max())
    rel_u  = np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title, fontsize=10)
    u_perc_err = 100 * np.abs(u_pred - u_true) / (np.abs(u_true) + 1e-12)
    u_perc_err_clipped = np.clip(u_perc_err, 0, 10) # Cap at 10%

    panels = [
        (axes[0], u_true,                  "jet",   "True u [µm/s]",            vmin_u, vmax_u),
        (axes[1], u_pred,                  "jet",   "Predicted u [µm/s]",        vmin_u, vmax_u),
        (axes[2], u_perc_err_clipped, "magma", f"% error u (Sample Rel. Err.={rel_u:.2e})", 0, 10),
    ]

    for ax, data, cmap, ptitle, vlo, vhi in panels:
        sc = ax.scatter(x_np, y_np, c=data, s=3, cmap=cmap, vmin=vlo, vmax=vhi)
        add_pillars(ax)
        plt.colorbar(sc, ax=ax)
        ax.set_title(ptitle, fontsize=9)
        ax.set_xlabel("x [µm]");  ax.set_ylabel("y [µm]")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

## 18. Show 3 prediction examples # CLAUDE CODED

example_indices = sorted(set([len(all_files)//100, len(all_files)//2, len(all_files)//3]))

for i in example_indices:
    path = all_files[i]
    xy, uv_true, uv_pred, theta = predict_on_file(model, path)
    rel   = torch.norm(uv_pred - uv_true) / (torch.norm(uv_true) + 1e-12)
    label = os.path.basename(path)
    print(f"\n[{i}] {label}")
    print(f"     theta=[y1,y2]={theta.numpy()}  |  global rel. error={rel.item():.3e}")

    save_path = os.path.join(DRIVE_SAVE_DIR, f"pred_ex{i}_{RUN_NAME}.png")
    plot_prediction(
        xy, uv_true, uv_pred, theta,
        title=(
            f"Example {i}: {label}\n"
            f"[y1={theta[0]:.3e}, y2={theta[1]:.3e}]  —  rel. err={rel.item():.3e}\n"
            f"{param_str}"
        ),
        save_path=save_path
    )

## 19. TEST eval

model.eval()
relative_l2_errors = []

with torch.no_grad():
    for theta_raw, xy_raw, uv_true_raw, sdf_raw in test_loader:
        theta_raw = theta_raw.to(device)
        xy_raw = xy_raw.to(device)
        uv_true_raw = uv_true_raw.to(device)
        sdf_raw = sdf_raw.to(device)

        theta_norm = torch.stack([normalise_theta(t) for t in theta_raw])
        xy_norm = torch.stack([normalise_xy(x)    for x in xy_raw])
        uv_norm = normalise_uv(uv_true_raw)
        sdf_norm = sdf_raw / L_y

        trunk_input = torch.cat([xy_norm, sdf_norm.unsqueeze(-1)], dim=-1)
        uv_pred = model(theta_norm, trunk_input)

        for i in range(uv_pred.shape[0]):
            error_norm = torch.norm(uv_pred[i] - uv_norm[i])
            true_norm = torch.norm(uv_norm[i])
            if true_norm > 1e-12:
                relative_l2_errors.append((error_norm / true_norm).item())
            else:
                relative_l2_errors.append(0.0)


mean_relative_l2_error = np.mean(relative_l2_errors)
std_relative_l2_error = np.std(relative_l2_errors)

print(f"Mean relative L2 error (test): {mean_relative_l2_error:.3e}")
print(f"Std  relative L2 error (test) {std_relative_l2_error:.3e}")


## 20. Save checkpoint

checkpoint_path = os.path.join(DRIVE_SAVE_DIR, f"{RUN_NAME}.pt")

torch.save({
    "model_state"  : model.state_dict(),
    "dot_dim"      : DOT_DIM,
    "hidden"       : HIDDEN,
    "epochs"       : EPOCHS,
    "lr"           : LR,
    "W_DATA"       : W_DATA,
    "W_WALL"       : W_WALL,
    "L_x"          : L_x,
    "L_y"          : L_y,
    "VEL_SCALE"    : VEL_SCALE,
    "X_ROW1"   : X_ROW1,
    "R_SCALE"  : R_SCALE,
    "history"      : history,
    "run_name"     : RUN_NAME,}, checkpoint_path)

print(f"\nCheckpoint saved:\n  {checkpoint_path}")